# Read an Avro file in Python with chDB (drop-in pandas)

Companion to [read-avro-file-python](https://clickhouse.com/resources/engineering/read-avro-file-python).

Run `./generate.sh` first to create `data/`. Requires `pip install chdb pandas fastavro`.

## 1. Read an Avro file into a DataFrame (types auto-inferred)

In [ ]:
from chdb.datastore import DataStore

df = DataStore.from_file("data/events.avro", format="Avro")
df

In [ ]:
df.dtypes

## 2. Filter + aggregate the way you already do (pandas, not SQL)

In [ ]:
revenue = df[df["event_type"] == "purchase"].groupby("country")["amount"].sum()
revenue

## 3. Hand off to real pandas when you need it

In [ ]:
pdf = df.to_pandas()   # a real pandas.DataFrame, in memory
type(pdf)

## 4. Performance: DataStore vs fastavro + manual loop, on a 3M-row Avro file

Apple M4 Pro (14 cores, 24 GB RAM, macOS), best-of-3 warm: ~5.7x faster than a fastavro reader loop.

In [ ]:
import time
import fastavro
from collections import defaultdict

def datastore_agg():
    d = DataStore.from_file("data/events_large.avro", format="Avro")
    r = (d[d["event_type"] == "purchase"]
         .groupby("country")["amount"].sum()
         .sort_values(ascending=False))
    return r.to_pandas()

def fastavro_agg():
    agg = defaultdict(lambda: [0, 0.0])
    with open("data/events_large.avro", "rb") as f:
        for r in fastavro.reader(f):
            if r["event_type"] == "purchase":
                a = agg[r["country"]]; a[0] += 1; a[1] += r["amount"]
    return agg

def best_of_3(fn):
    fn()
    best = float("inf")
    for _ in range(3):
        t0 = time.perf_counter(); fn(); best = min(best, time.perf_counter() - t0)
    return best

fa_s = best_of_3(fastavro_agg)
ds_s = best_of_3(datastore_agg)
print(f"fastavro (manual loop):         {fa_s:.3f}s")
print(f"import chdb.datastore as pd:    {ds_s:.3f}s")
print(f"speedup:                        {fa_s / ds_s:.1f}x")